In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bson.json_util import dumps
import re # Needed for Regular Expression searching since Mongo does not support it natively

# Import custtom CRUD Module to access database
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "cs340"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
# df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)


image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Creating the Layout for the entire dashboard
app.layout = html.Div([
    # Creating the header with Company Logo and Unique Identifier
    html.A([html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), height = 250, width = 250))], href = 'www.snhu.edu', target = "_blank"),
    html.Center(html.B(html.H1('CS-340 Dashboard - Asaro, Vincent'))),
    html.Hr(),

    # Setting up the radio buttons to use as filters
    html.Div(
        dcc.RadioItems(
            id = 'filter-type',
            options = [
                {'label':'All', 'value':'All'},
                {'label':'Water Rescue', 'value':'Water'},
                {'label':'Mountain or Wilderness Rescue', 'value':'Mountain'},
                {'label':'Disaster Rescue or Individual Tracking', 'value':'Disaster'},
            ],
            value = 'All'
        )
    ),
    html.Hr(),

    # Setting up the data table
    dash_table.DataTable(
        id = 'datatable-id',
        columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data = df.to_dict('records'),
        editable = True,
        row_selectable = "single",
        selected_rows = [],
        filter_action = "native",
        sort_action = "native",
        page_action = "native",
        page_current = 0,
        page_size = 10,
    ),
    html.Br(),
    html.Hr(),

#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id','data'),
              Output('datatable-id', 'columns'),
              [Input('filter-type', 'value')])

# Updates the data table based on which preset filter is used
def update_dashboard(filter_type):
    if filter_type == 'All':
        df = pd.DataFrame.from_records(db.read({}))
    elif filter_type == 'Water':
        labRegex = re.compile(".*lab.*", re.IGNORECASE)
        chesaRegex = re.compile(".*chesa.*", re.IGNORECASE)
        newfRegex = re.compile(".*newf.*", re.IGNORECASE)
        
        df = pd.DataFrame.from_records(db.read({
            '$or':[
                {"breed":{"$regex": labRegex}},
                {"breed":{"$regex": chesaRegex}},
                {"breed":{"$regex": newfRegex}},
            ],
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks":{"$gte":26.0, "$lte":156.0}
        }))
    elif filter_type == 'Mountain':
        germanRegex = re.compile(".*german.*", re.IGNORECASE)
        malaRegex = re.compile(".*mala.*", re.IGNORECASE)
        oldRegex = re.compile(".*old english.*", re.IGNORECASE)
        huskyRegex = re.compile(".*husk.*", re.IGNORECASE)
        rottRegex = re.compile(".*rott.*", re.IGNORECASE)
        
        df = pd.DataFrame.from_records(db.read({
            '$or':[
                {"breed":{"$regex": germanRegex}},
                {"breed":{"$regex": malaRegex}},
                {"breed":{"$regex": oldRegex}},
                {"breed":{"$regex": huskyRegex}},
                {"breed":{"$regex": rottRegex}},
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks":{"$gte":26.0, "$lte":156.0}
        }))
    elif filter_type == 'Disaster':
        germanRegex = re.compile(".*german.*", re.IGNORECASE)
        goldenRegex = re.compile(".*golden.*", re.IGNORECASE)
        bloodRegex = re.compile(".*blood.*", re.IGNORECASE)
        doberRegex = re.compile(".*dober.*", re.IGNORECASE)
        rottRegex = re.compile(".*rott.*", re.IGNORECASE)
        
        df = pd.DataFrame.from_records(db.read({
            '$or':[
                {"breed":{"$regex": germanRegex}},
                {"breed":{"$regex": goldenRegex}},
                {"breed":{"$regex": bloodRegex}},
                {"breed":{"$regex": doberRegex}},
                {"breed":{"$regex": rottRegex}},
            ],
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks":{"$gte":20.0, "$lte":300.0}
        }))
    else:
        raise Exception("Unknown Filter.")
    
    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data = df.to_dict('records')
    
    return (data, columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

# Updates the pie chart depending on what preset filter is used
def update_graphs(viewData):
    
    dffPie = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(            
            figure = px.pie(dffPie, names='breed',)
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_selected_rows")])

# Updates the map with the correct location for a selected animal
def update_map(virtualRows):  
    
    if not virtualRows:
        markerArray = (30.75, -97.48) # Austin TX is at [30.75,-97.48]
        toolTip = "Austin Animal Center"
        popUpHeading = "Austin Animal Center"
        popUpParagraph = "Shelter Home Location"
    else:
        dff = pd.DataFrame(df.iloc[virtualRows])
        coordLat = float(dff['location_lat'].to_string().split()[1])
        coordLong = float(dff['location_long'].to_string().split()[1])
        markerArray = (coordLat, coordLong)
        
        toolTip = dff['breed']
        popUpHeading = "Animal Name"
        popUpParagraph = dff['name']
        
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=markerArray, zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=markerArray, children=[
                dl.Tooltip(toolTip),
                dl.Popup([
                    html.H1(popUpHeading),
                    html.P(popUpParagraph)
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://snowspark-investschool-3000.codio.io/proxy/8050/
